In [ ]:
import numpy as np
import pandas as pd
from lsst.daf.butler import Butler

In [ ]:
instrument = "LSSTCam"
butler = Butler('main')   # fresh instance: sees collections created after the notebook started
base = 'u/dagoret/LSSTCam/runelaiss1_20261111_tract2877patchsel_mediumpatcharray'
for c in sorted(butler.registry.queryCollections(f'{base}/*')):
    print(c)

In [ ]:
collection_path = f'{base}/stage2_v2/20261112T110933Z'

In [ ]:
# fgcmBuildFromIsolatedStar succeeded, so its outputs are in this chain
obs = butler.get('fgcm_star_observations', instrument=instrument, collections=collection_path)
vcat = butler.get('fgcmVisitCatalog', instrument=instrument, collections=collection_path)
print("--------------------------------------------------------")
print(len(obs), 'observations; columns:', list(obs.colnames))

# visit -> band, epoch (same epochMjds as in config_fgcmFitCycle.py)
band_of = {r.name: r.band for r in butler.registry.queryDimensionRecords('physical_filter', instrument=instrument)}
visits = pd.DataFrame([{'visit': rec['visit'], 'mjd': rec['mjd'], 'physicalFilter': rec['physicalFilter']}
                       for rec in vcat])
visits['band'] = visits['physicalFilter'].map(band_of)
visits['epoch'] = np.digitize(visits['mjd'], [60859.0, 60949.0])

o = pd.DataFrame({'visit': np.asarray(obs['visit']), 'detector': np.asarray(obs['detector'])})
o = o.merge(visits[['visit', 'band', 'epoch']], on='visit')
print("--------------------------------------------------------")
print('detectors with at least one observation (out of 189):', o['detector'].nunique())

# number of distinct detectors carrying stars, per (band, epoch)
print("--------------------------------------------------------")
print(o.groupby(['band', 'epoch'])['detector'].nunique().unstack().reindex(list('ugrizy')))

# observations per detector, per (band, epoch): few observations per CCD means an ill-constrained superstar
per_det = o.groupby(['band', 'epoch', 'detector']).size().rename('n_obs')
print("--------------------------------------------------------")
print(per_det.groupby(['band', 'epoch']).describe(percentiles=[.1, .5]).round(0))

In [ ]:
cols = ['inst_mag', 'inst_mag_err', 'delta_mag_bkg', 'delta_mag_aper', 'delta_mag_err_aper', 'x', 'y']
o = pd.DataFrame({c: np.asarray(obs[c]) for c in cols + ['visit', 'detector']})
o = o.merge(visits[['visit', 'band', 'epoch']], on='visit')
key = [o['band'], o['epoch']]
print("--------------------------------------------------------")
print('non-finite values per column:')
print((~np.isfinite(o[cols])).groupby(key).sum().to_string())
print("--------------------------------------------------------")
print('values equal to -9999 per column:')
print((o[cols] == -9999).groupby(key).sum().to_string())
print("--------------------------------------------------------")
print(o.groupby(['band', 'epoch']).agg(
    n=('visit', 'size'),
    imag_min=('inst_mag', 'min'), imag_max=('inst_mag', 'max'),
    daper_min=('delta_mag_aper', 'min'), daper_max=('delta_mag_aper', 'max'),
    dbkg_min=('delta_mag_bkg', 'min'), dbkg_max=('delta_mag_bkg', 'max'),
    daper_err_max=('delta_mag_err_aper', 'max')).round(3).to_string())

In [ ]:
tot = o.groupby(['band', 'detector']).size().unstack('band').reindex(columns=list('ugrizy'))
print('detectors with no observation in some band:'); print(tot[tot.isna().any(axis=1)])
for b in 'ugrizy':
    print(b, 'fewest observations:', tot[b].dropna().nsmallest(4).astype(int).to_dict())